In [2]:
!pip install -q langgraph langchain-openai langchain-chroma langchain-huggingface sentence-transformers langchain-community

In [5]:
import os
import getpass
from typing import List, TypedDict
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END

# 1. OpenAI API Key 설정 (이미 설정되어 있다면 건너뜀)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

# 2. 임베딩 모델 & DB 로드 (검색 기능 유지)
print("--- 임베딩 모델 및 DB 로드 중... ---")
hf_embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="./chroma_db_kt_terms",
    embedding_function=hf_embeddings,
    collection_name="kt_terms"
)

# ==========================================
# 3. 상태(State) 정의
# ==========================================
class GraphState(TypedDict):
    summary: str            # 입력: 상담원-고객 대화 요약본
    search_query: str       # 중간산출물: 추출된 검색 키워드
    documents: List[Document] # 결과: 검색된 문서들

# ==========================================
# 4. 노드(Node) 정의 - 핵심 역할 집중!
# ==========================================

# [Node 1] 대화 요약 분석 및 키워드 추출 (여기가 핵심 역량)
def query_analyzer_node(state: GraphState):
    summary = state["summary"]
    print(f"\n📊 [분석 중] 상담 요약 내용: {summary}")

    # 비용 효율적인 gpt-5-nano 사용
    llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

    # 프롬프트: 상담 내용을 분석하여 약관 검색에 필요한 '전문 용어'나 '키워드'로 변환
    prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 고객 상담 내용을 분석하여 관련 약관을 찾아주는 검색 전문가입니다.
        제공된 [상담 대화 요약]을 읽고, 약관 데이터베이스(Vector DB)에서 검색할 때 가장 정확도가 높을 '핵심 키워드 3~5개'를 띄어쓰기로 구분하여 추출하세요.

        예시:
        입력: 고객이 이사로 인해 인터넷을 이전 설치하려고 하는데 비용이 발생하는지 문의함
        출력: 인터넷 이전 설치비 댁내 이전 출동비 면제 조건
        """),
        ("user", "{summary}")
    ])

    response = (prompt | llm).invoke({"summary": summary})
    refined_query = response.content

    print(f"🔑 [키워드 추출 완료] -> '{refined_query}'")
    return {"search_query": refined_query}

# [Node 2] 문서 검색 (LLM 없이 순수 검색)
def retriever_node(state: GraphState):
    query = state["search_query"]
    print(f"📚 [DB 검색 수행] 키워드: {query}")

    # 검색 (정확도를 위해 k=3 유지)
    results = vectorstore.similarity_search(query, k=3)
    return {"documents": results}

# ==========================================
# 5. 결과 포맷팅 함수 (AI 대신 깔끔하게 다듬는 역할)
# ==========================================
def format_docs(docs: List[Document]):
    """
    검색된 문서 리스트를 보기 좋게 정제하여 문자열로 반환합니다.
    """
    formatted_output = ""
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "약관 파일").split("/")[-1] # 경로 제외하고 파일명만
        page = doc.metadata.get("page", 0) + 1
        content = doc.page_content.replace('\n', ' ').strip()

        # 가독성을 위해 텍스트가 너무 길면 적당히 자름 (선택사항)
        # content = content[:300] + "..." if len(content) > 300 else content

        formatted_output += f"""
        📄 [문서 {i+1}] {source} (p.{page})
        ────────────────────────────────────────
        {content}
        ────────────────────────────────────────
        """
    return formatted_output

# ==========================================
# 6. 그래프(Graph) 연결
# ==========================================
workflow = StateGraph(GraphState)

# 노드 추가
workflow.add_node("analyze_query", query_analyzer_node)
workflow.add_node("retrieve_docs", retriever_node)

# 엣지 연결 (분석 -> 검색 -> 종료)
workflow.add_edge(START, "analyze_query")
workflow.add_edge("analyze_query", "retrieve_docs")
workflow.add_edge("retrieve_docs", END)

app = workflow.compile()

# ==========================================
# 7. 실행 테스트
# ==========================================
def run_consulting_search(conversation_summary):
    inputs = {"summary": conversation_summary}
    result = app.invoke(inputs)

    # 결과 출력
    print("\n" + "="*60)
    print(f"🗣️ 입력된 상담 요약: {conversation_summary}")
    print("-" * 60)
    print(f"🗝️ AI가 추출한 검색 키워드: {result['search_query']}")
    print("-" * 60)
    print("📄 검색된 약관 조항 (Raw Data 정제본):")
    print(format_docs(result['documents']))
    print("=" * 60)

# 테스트 시나리오: 실제 상담 요약처럼 입력
summary_input = "고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다."
run_consulting_search(summary_input)

--- 임베딩 모델 및 DB 로드 중... ---

📊 [분석 중] 상담 요약 내용: 고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다.
🔑 [키워드 추출 완료] -> '중도해지 위약금 잔여기간 2년약정 해지수수료'
📚 [DB 검색 수행] 키워드: 중도해지 위약금 잔여기간 2년약정 해지수수료

🗣️ 입력된 상담 요약: 고객이 현재 2년 약정으로 인터넷을 쓰고 있는데, 1년 만에 해지하면 위약금이 얼마나 나오는지 계산하는 공식을 궁금해하십니다.
------------------------------------------------------------
🗝️ AI가 추출한 검색 키워드: 중도해지 위약금 잔여기간 2년약정 해지수수료
------------------------------------------------------------
📄 검색된 약관 조항 (Raw Data 정제본):

        📄 [문서 1] (이용약관전문)인터넷서비스이용약관_202509.pdf (p.155)
        ────────────────────────────────────────
        ※ 추가 계약기간 이내 해지하거나 계약기간을 단축할 시는 추가 할인된 요금을 반환하여야 함  - A형 산정식 : (약정기간 할인금 - 사용기간 할인금) x 경과월수  - B형 산정식 : (할인전 월 이용료×경과월수)×(계약기간 할인율-사용기간 할인율)  - 사용기간 할인율(또는 할인금)은 1년 미만은 무약정, 2년 미만은 1년 약정, 3년 미만은 2년 약정, 4년 미만은 3년 약정기간 할인  율(또는 할인금)을 적용함  ※ 계약기간에 따른 할인율을 적용받는 이용고객이 만료 시까지 해지 의사 표시를 하지 않는 경우, 해지 의사를 표시할 때까지 기존  계약 내용과 동일한 내용으로 계약이 연장됨  ※ 계약기간 만료 이후 계약이 연장된 고객은 서비스를 해지하더라도 할인반